<a href="https://colab.research.google.com/github/antonypradeep54/Machine-Learning---NYC-taxi-trip-duration-prediction/blob/main/NYC_Taxi_Trip_Duration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NYC Taxi Trip Duration Prediction


## Table of contents

- Problem statement
- Libraries and functions
- Import dataset
- Data cleaning
- Hypothesis test (t-test)
- Regression model (Random Forest)
- References



##Problem Statement

The NYC Taxi Trip Duration dataset provides detailed records of taxi trips, including pickup and drop-off times, locations, and other relevant features. The objective of this project is to develop a predictive model to estimate the trip duration based on these attributes. Accurate predictions can enhance ride-hailing services, optimize route planning, and improve overall transportation efficiency in New York City.

<a id="Librarys"></a>

## Libraries and functions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path


<a id="ImportDatasets"></a>

## Import dataset

In [ ]:
# Colab users: upload the CSV when it is not already available.
from pathlib import Path
DATA_PATH = Path("dataset/nyc_taxi_trip_duration.csv")
if not DATA_PATH.exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(f"Dataset missing at {DATA_PATH}; see README for setup.") from exc
    uploaded = files.upload()
    if "nyc_taxi_trip_duration.csv" not in uploaded:
        raise FileNotFoundError("Upload nyc_taxi_trip_duration.csv to continue.")
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    DATA_PATH.write_bytes(uploaded["nyc_taxi_trip_duration.csv"])
print(f"Dataset path: {DATA_PATH}")

In [ ]:
# Check dataset availability
assert DATA_PATH.is_file(), f"Missing dataset: {DATA_PATH}"

In [ ]:
# No manual file moves are necessary; the loader above handles Colab uploads.
print("Dataset ready.")

In [ ]:
NYC_Taxi_df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(NYC_Taxi_df):,} rows and {len(NYC_Taxi_df.columns)} columns")
NYC_Taxi_df.head()

<a id="Data Cleaning"></a>

## Data Cleaning

In [ ]:
NYC_Taxi_df.dtypes

#### Following columns have data type as object which needs to be changed to datetime. </br>

*   pickup_datetime
*   dropoff_datetime



In [ ]:
# Convert pickup_datetime and dropoff_datetime columns to datetime
NYC_Taxi_df['pickup_datetime'] = pd.to_datetime(NYC_Taxi_df['pickup_datetime'])
NYC_Taxi_df['dropoff_datetime'] = pd.to_datetime(NYC_Taxi_df['dropoff_datetime'])

In [ ]:
# Adding few additional columns based on pickup_datetime and dropoff_datetime columns for further analysis

NYC_Taxi_df['pickup_hour'] = NYC_Taxi_df['pickup_datetime'].dt.hour
NYC_Taxi_df['pickup_day'] = NYC_Taxi_df['pickup_datetime'].dt.day
NYC_Taxi_df['day_of_week'] = NYC_Taxi_df['pickup_datetime'].dt.dayofweek
NYC_Taxi_df['pickup_weekday'] = NYC_Taxi_df['pickup_datetime'].dt.weekday  # 0=Monday, 6=Sunday
NYC_Taxi_df['pickup_month'] = NYC_Taxi_df['pickup_datetime'].dt.month

NYC_Taxi_df['dropoff_hour'] = NYC_Taxi_df['dropoff_datetime'].dt.hour
NYC_Taxi_df['dropoff_day'] = NYC_Taxi_df['dropoff_datetime'].dt.day
NYC_Taxi_df['dropoff_weekday'] = NYC_Taxi_df['dropoff_datetime'].dt.weekday  # 0=Monday, 6=Sunday
NYC_Taxi_df['dropoff_month'] = NYC_Taxi_df['dropoff_datetime'].dt.month



In [ ]:
NYC_Taxi_df.head()

In [ ]:
NYC_Taxi_df.dtypes

#### Applying mapping on follwoing to convert from numerical values to actual weekday and month respectively. </br>

*   pickup_weekday
*   dropoff_weekday
*   pickup_month
*   dropoff_month

In [ ]:
# Create a dictionary to perfrom required weekday mapping
weekday_mapping = {
    0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday',
    4: 'Friday', 5: 'Saturday', 6: 'Sunday'
}

# Convert numeric weekday to weekday name
NYC_Taxi_df['pickup_weekday_mapped'] = NYC_Taxi_df['pickup_weekday'].map(weekday_mapping)
NYC_Taxi_df['dropoff_weekday_mapped'] = NYC_Taxi_df['dropoff_weekday'].map(weekday_mapping)

In [ ]:
# Create a dictionary to perfrom required month mapping
month_mapping = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April',
    5: 'May', 6: 'June', 7: 'July', 8: 'August',
    9: 'September', 10: 'October', 11: 'November', 12: 'December'
}

# Convert numeric weekday to weekday name
NYC_Taxi_df['pickup_month_mapped'] = NYC_Taxi_df['pickup_month'].map(month_mapping)
NYC_Taxi_df['dropoff_month_mapped'] = NYC_Taxi_df['dropoff_month'].map(month_mapping)

In [ ]:
NYC_Taxi_df.head()

<a id="HypothesisTest"></a>

## Hypothesis Test


<a id="Ttest"></a>

## t-Test

t-test compares the means of two independent groups in order to determine whether there is statistical evidence that the associated population means are significantly different. Null hypothesis and Alterntive hypothesis for T-test is given below. </br>

* Null hypothesis: The means in the two groups are equal (so there is no difference between the two groups) </br>
* Alternative hypothesis: The mean values in the two groups are not equal (i.e. there is a difference between the two groups) </br>

Note : If the calcualted p-value is below significance level (typically p < 0.05), then null hypothesis is rejected.

In [ ]:
# Separate the data for the two vendors
vendor_1_data = NYC_Taxi_df[NYC_Taxi_df['vendor_id'] == 1]['trip_duration']
vendor_2_data = NYC_Taxi_df[NYC_Taxi_df['vendor_id'] == 2]['trip_duration']

# Perform the t-test
t_stat, p_value = stats.ttest_ind(vendor_1_data, vendor_2_data)

# Output the results
print(f"P-value: {p_value}")

#### Insight : t-test

* Since the calcualted p-value is less than 5, the Null hypothesis is rejected and the Alternative hypothesis is accepted. </br>
* The mean trip duration for Vendor 1 and Vendor 2 are not equal.


#Regression model (Random Forest)

In [ ]:
def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Radius of earth in km
    return c * r


NYC_Taxi_df['distance'] = haversine(NYC_Taxi_df['pickup_longitude'], NYC_Taxi_df['pickup_latitude'],
                           NYC_Taxi_df['dropoff_longitude'], NYC_Taxi_df['dropoff_latitude'])

# Log transform target
NYC_Taxi_df['log_trip_duration'] = np.log1p(NYC_Taxi_df['trip_duration'])

In [ ]:
# Validate the target without silently removing outliers.
target = pd.to_numeric(NYC_Taxi_df["trip_duration"], errors="coerce")
assert target.notna().all(), "trip_duration contains missing or non-numeric values"
assert np.isfinite(target.to_numpy(dtype=float)).all(), "trip_duration contains non-finite values"
assert (target > 0).all(), "trip_duration must be positive"
print("Duration quantiles (seconds):")
print(target.quantile([0, 0.50, 0.75, 0.95, 0.99, 0.999, 1.0]))
long_trip_count = int((target > 6 * 60 * 60).sum())
print(f"Trips longer than 6 hours: {long_trip_count:,}")
print("Flagged for investigation only; no rows removed.")

In [ ]:
# Haversine distance was already computed with vectorized NumPy in the previous cell.
# Do not repeat the calculation using the slower DataFrame.apply().
assert np.isfinite(NYC_Taxi_df["distance"]).all(), "Non-finite distance values"

In [ ]:
# Features & Target
features = ['passenger_count', 'pickup_hour', 'day_of_week', 'distance']
X = NYC_Taxi_df[features]
y = NYC_Taxi_df['log_trip_duration']

In [ ]:
from sklearn.model_selection import train_test_split

# Train/val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_log_error, mean_absolute_error

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred_log = model.predict(X_val)
y_val_seconds = np.expm1(y_val.to_numpy())
y_pred_seconds = np.clip(np.expm1(y_pred_log), 0, None)
rmsle = np.sqrt(mean_squared_log_error(y_val_seconds, y_pred_seconds))
mae_seconds = mean_absolute_error(y_val_seconds, y_pred_seconds)
print(f"Validation RMSLE (original duration scale): {rmsle:.6f}")
print(f"Validation MAE (seconds): {mae_seconds:.1f}")

## Package 03 — Baseline and chronological validation
The baseline is fitted only on training data. Both models are evaluated on the same holdout for each split. The chronological holdout is a separate experiment; it does not overwrite the original random-split result.

In [ ]:
# Package 03: fair baseline on the SAME random validation split.
# Predict the training-set median duration for every validation record.
baseline_seconds = float(np.median(np.expm1(y_train.to_numpy())))
baseline_pred_seconds = np.full(len(y_val_seconds), baseline_seconds, dtype=float)
baseline_rmsle = np.sqrt(mean_squared_log_error(y_val_seconds, baseline_pred_seconds))
baseline_mae_seconds = mean_absolute_error(y_val_seconds, baseline_pred_seconds)
print(f"Median baseline RMSLE: {baseline_rmsle:.6f}")
print(f"Median baseline MAE (minutes): {baseline_mae_seconds / 60:.2f}")
print(f"Random Forest RMSLE: {rmsle:.6f}")
print(f"Random Forest MAE (minutes): {mae_seconds / 60:.2f}")

In [ ]:
# Chronological holdout: train on earlier pickups and validate on later pickups.
# Keep the original random-split model intact for an apples-to-apples reference.
from sklearn.base import clone

chronological_order = NYC_Taxi_df["pickup_datetime"].sort_values(kind="stable").index
cutoff = int(len(chronological_order) * 0.8)
early_index = chronological_order[:cutoff]
late_index = chronological_order[cutoff:]
assert NYC_Taxi_df.loc[early_index, "pickup_datetime"].max() <= NYC_Taxi_df.loc[late_index, "pickup_datetime"].min()
X_train_time, X_val_time = X.loc[early_index], X.loc[late_index]
y_train_time, y_val_time = y.loc[early_index], y.loc[late_index]
time_model = clone(model)
time_model.fit(X_train_time, y_train_time)
time_pred_seconds = np.clip(np.expm1(time_model.predict(X_val_time)), 0, None)
time_actual_seconds = np.expm1(y_val_time.to_numpy())
time_rmsle = np.sqrt(mean_squared_log_error(time_actual_seconds, time_pred_seconds))
time_mae_seconds = mean_absolute_error(time_actual_seconds, time_pred_seconds)
time_baseline_seconds = float(np.median(np.expm1(y_train_time.to_numpy())))
time_baseline_pred = np.full(len(time_actual_seconds), time_baseline_seconds)
print(f"Chronological holdout: {len(X_train_time):,} training / {len(X_val_time):,} validation")
print(f"Chronological median baseline RMSLE: {np.sqrt(mean_squared_log_error(time_actual_seconds, time_baseline_pred)):.6f}")
print(f"Chronological median baseline MAE (minutes): {mean_absolute_error(time_actual_seconds, time_baseline_pred) / 60:.2f}")
print(f"Chronological Random Forest RMSLE: {time_rmsle:.6f}")
print(f"Chronological Random Forest MAE (minutes): {time_mae_seconds / 60:.2f}")

In [ ]:
# Diagnose the magnitude of errors on the chronological holdout.
absolute_errors_minutes = np.abs(time_actual_seconds - time_pred_seconds) / 60
print("Chronological absolute-error quantiles (minutes):")
print(pd.Series(absolute_errors_minutes).quantile([0.50, 0.90, 0.95, 0.99]))
print(f"Trips with absolute error > 15 minutes: {(absolute_errors_minutes > 15).mean():.1%}")
print("These diagnostics are descriptive, not evidence of production accuracy.")

Benchmarks for Root Mean Squared Logarithmic Error

Less than 0.1 → Excellent

Between 0.1 – 0.2 → Good

Between 0.2 – 0.5 → Moderate / acceptable depending on use case

Greater than 0.5 → Poor


In [ ]:
print(f"Validation RMSLE (original duration scale): {rmsle:.6f}")
print(f"Validation MAE (minutes): {mae_seconds / 60:.2f}")
print("Results must be rerun and recorded before external publication.")

<a id="References"></a>

## References

* ChatGPT for Python syntax. </br>
* DataTab for Statistics. </br>


<font size=+1 color="blue"><center><b> End of Assignment </b></center></font>

## Execution environment and validation summary

In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Total dataset records:", len(NYC_Taxi_df))
print("Training records:", len(X_train))
print("Validation records:", len(X_val))
print("Model:", model)
print("RMSLE:", round(rmsle, 6))
print("MAE (seconds):", round(mae_seconds, 2))
print("MAE (minutes):", round(mae_seconds / 60, 2))